# Ward/Season Inference Pipeline

Three-step SageMaker Pipeline for ward-level livestock mortality risk inference.

**Steps**
1. `InferencePreprocess` — feature selection, median imputation, Ridge scaling
2. `ModelInference`     — 4-model ensemble with normalized Spearman weights
3. `InferencePostprocess` — ward-level risk levels, confidence, SHAP top features, GeoJSON + GeoTIFF

**Parameters**
- `InputDataS3Path`  — S3 URI to pre-aggregated ward+season parquet
- `ModelS3Prefix`   — S3 prefix containing `biannual/`, `quadseasonal/`, `monthly/` artifact folders
- `SeasonScheme`    — one of `biannual`, `quadseasonal`, `monthly`
- `OutputS3Prefix`  — base S3 URI where all outputs are written

In [ ]:
import sagemaker
from sagemaker.workflow.execution_variables import ExecutionVariables
from sagemaker.workflow.function_step import step
from sagemaker.workflow.parameters import ParameterString
from sagemaker.workflow.pipeline import Pipeline

from inference_config import S3_BUCKET, MODEL_BASE_PREFIX, WARD_BOUNDARIES_S3_KEY

In [ ]:
sagemaker_session = sagemaker.session.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session.boto_region_name

pipeline_name  = "lmr-ward-inference-pipeline"
instance_type  = ParameterString(name="InferenceInstanceType", default_value="ml.m5.xlarge")

# MLflow tracking (same server as training pipeline)
tracking_server_arn = (
    "arn:aws:sagemaker:us-east-1:575108933641:"
    "mlflow-tracking-server/lmr-tracking-server-5t7l23o0xvt99j-chws71x3trpelj-dev"
)
experiment_name = "lmr-ward-inference"

## Pipeline Parameters

In [ ]:
InputDataS3Path = ParameterString(
    name="InputDataS3Path",
    default_value=(
        f"s3://{S3_BUCKET}/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/data/inference/"
        "ward_season_input.parquet"
    ),
)

ModelS3Prefix = ParameterString(
    name="ModelS3Prefix",
    default_value=f"s3://{S3_BUCKET}/{MODEL_BASE_PREFIX}",
)

SeasonScheme = ParameterString(
    name="SeasonScheme",
    default_value="biannual",
)

OutputS3Prefix = ParameterString(
    name="OutputS3Prefix",
    default_value=(
        f"s3://{S3_BUCKET}/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/outputs/"
        "inference-outputs"
    ),
)

## Step Definitions

In [ ]:
@step(
    name="InferencePreprocess",
    instance_type=instance_type,
)
def inference_preprocess(
    input_data_s3_path: str,
    model_s3_prefix: str,
    season_scheme: str,
    output_s3_base_uri: str,
):
    """
    Returns: (features_s3, features_ridge_s3, metadata_s3, label_mean)
    """
    from inference_preprocess import run_inference_preprocess
    return run_inference_preprocess(
        input_data_s3_path=input_data_s3_path,
        model_s3_prefix=model_s3_prefix,
        season_scheme=season_scheme,
        output_s3_base_uri=output_s3_base_uri,
    )

In [ ]:
@step(
    name="ModelInference",
    instance_type=instance_type,
)
def model_inference(
    features_s3: str,
    features_ridge_s3: str,
    metadata_s3: str,
    model_s3_prefix: str,
    season_scheme: str,
    output_s3_base_uri: str,
):
    """
    Returns: predictions_s3
    """
    from inference import run_inference
    return run_inference(
        features_s3=features_s3,
        features_ridge_s3=features_ridge_s3,
        metadata_s3=metadata_s3,
        model_s3_prefix=model_s3_prefix,
        season_scheme=season_scheme,
        output_s3_base_uri=output_s3_base_uri,
    )

In [ ]:
@step(
    name="InferencePostprocess",
    instance_type=instance_type,
)
def inference_postprocess(
    predictions_s3_path: str,
    features_s3: str,
    metadata_s3: str,
    model_s3_prefix: str,
    season_scheme: str,
    output_s3_prefix: str,
    training_label_mean: float,
    experiment_name: str,
    run_id: str,
):
    """
    Returns: (ward_csv_s3, ward_geojson_s3, ward_geotiff_s3)
    """
    import boto3
    import json
    import os
    import tempfile
    import joblib
    import pandas as pd
    import shap as shap_lib
    from inference_config import S3_BUCKET, WARD_BOUNDARIES_S3_KEY
    from postprocess import run_postprocess

    s3 = boto3.client("s3")

    def _parse_s3(uri):
        parts = uri[5:].split("/", 1)
        return parts[0], parts[1] if len(parts) > 1 else ""

    with tempfile.TemporaryDirectory() as tmp:
        # Download ward boundary GeoJSON
        bounds_local = os.path.join(tmp, "boundaries.geojson")
        s3.download_file(S3_BUCKET, WARD_BOUNDARIES_S3_KEY, bounds_local)
        admin3_local_path = bounds_local

        # Download feature names + XGBoost model for SHAP
        bucket, key_prefix = _parse_s3(f"{model_s3_prefix}/{season_scheme}")
        fn_local = os.path.join(tmp, "feature_names.json")
        xgb_local = os.path.join(tmp, "xgboost_model.joblib")
        s3.download_file(bucket, f"{key_prefix}/feature_names.json", fn_local)
        s3.download_file(bucket, f"{key_prefix}/xgboost_model.joblib", xgb_local)
        with open(fn_local) as f:
            feature_names = json.load(f)
        xgb_model = joblib.load(xgb_local)

        # Load preprocessed features, metadata, and predictions
        X_raw = pd.read_parquet(features_s3)
        metadata_df = pd.read_parquet(metadata_s3)
        pred_df = pd.read_parquet(predictions_s3_path)

        # Compute SHAP values using XGBoost TreeExplainer
        try:
            explainer = shap_lib.TreeExplainer(xgb_model)
            shap_vals = explainer.shap_values(X_raw.values)
            shap_df = pd.DataFrame(shap_vals, columns=feature_names)
            shap_df["ward_name"] = metadata_df["ward_name"].values

            # Aggregate mean |SHAP| per ward across all seasons
            ward_shap = (
                shap_df.groupby("ward_name")[feature_names]
                .apply(lambda g: g.abs().mean())
            )

            def _top_features(row, n=5):
                top = row.nlargest(n)
                return json.dumps([
                    {"feature": feat, "importance": round(float(v), 6)}
                    for feat, v in top.items()
                ])

            ward_top = ward_shap.apply(_top_features, axis=1).reset_index()
            ward_top.columns = ["ward_name", "top_features"]

            # Merge pre-computed top_features into predictions df and overwrite
            pred_df = pred_df.merge(ward_top, on="ward_name", how="left")
            pred_df["top_features"] = pred_df["top_features"].fillna("[]")
            pred_df.to_parquet(predictions_s3_path, index=False)
            print(f"SHAP top features computed for {len(ward_top)} wards")
        except Exception as e:
            print(f"SHAP computation failed: {e}")

        return run_postprocess(
            predictions_s3_path=predictions_s3_path,
            experiment_name=experiment_name,
            run_id=run_id,
            training_run_id="",
            admin3_shapefile_path=admin3_local_path,
            prediction_column="prediction",
            feature_names=feature_names,
            top_n_features=5,
            output_s3_prefix=output_s3_prefix,
            granularity="ward",
            compute_shap=False,     # Already computed above via TreeExplainer
            training_label_mean=training_label_mean,
        )

## Wire Steps

In [ ]:
# Step 1 — preprocess
preprocess_step = inference_preprocess(
    input_data_s3_path=InputDataS3Path,
    model_s3_prefix=ModelS3Prefix,
    season_scheme=SeasonScheme,
    output_s3_base_uri=OutputS3Prefix,
)
# Returns: features_s3[0], features_ridge_s3[1], metadata_s3[2], label_mean[3]

# Step 2 — ensemble inference
inference_step = model_inference(
    features_s3=preprocess_step[0],
    features_ridge_s3=preprocess_step[1],
    metadata_s3=preprocess_step[2],
    model_s3_prefix=ModelS3Prefix,
    season_scheme=SeasonScheme,
    output_s3_base_uri=OutputS3Prefix,
)
# Returns: predictions_s3

# Step 3 — postprocess to ward-level GeoJSON + GeoTIFF
postprocess_step = inference_postprocess(
    predictions_s3_path=inference_step,
    features_s3=preprocess_step[0],
    metadata_s3=preprocess_step[2],
    model_s3_prefix=ModelS3Prefix,
    season_scheme=SeasonScheme,
    output_s3_prefix=OutputS3Prefix,
    training_label_mean=preprocess_step[3],
    experiment_name=experiment_name,
    run_id=ExecutionVariables.PIPELINE_EXECUTION_ID,
)
# Returns: ward_csv_s3[0], ward_geojson_s3[1], ward_geotiff_s3[2]

## Create and Upsert Pipeline

In [ ]:
pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        instance_type,
        InputDataS3Path,
        ModelS3Prefix,
        SeasonScheme,
        OutputS3Prefix,
    ],
    steps=[postprocess_step],  # SageMaker resolves the DAG automatically
    sagemaker_session=sagemaker_session,
)

pipeline.upsert(role_arn=role)
print(f"Pipeline '{pipeline_name}' upserted successfully.")

## Start Pipeline Execution

In [ ]:
execution = pipeline.start(
    parameters={
        # Override any defaults here, e.g.:
        # "SeasonScheme": "monthly",
        # "InputDataS3Path": "s3://...",
    }
)
print(f"Execution ARN: {execution.arn}")
execution.wait()